In [ ]:
# Importando bibliotecas

import pandas as pd
import cuml
import os
import scipy as sci
import time as tm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import decimate
import json

# Importando classificadores da biblioteca cuml
from cuml.ensemble import RandomForestClassifier #RF
from cuml.neighbors import KNeighborsClassifier #KNN
from cuml.svm import SVC #SVM
from cuml.naive_bayes import GaussianNB #GNB

# Importando PCA
from cuml.decomposition import PCA

# Importando separador dos dados, preprocessadores e métricas

from sklearn.model_selection import GridSearchCV, KFold, train_test_split, cross_val_score, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import ConfusionMatrixDisplay, f1_score, accuracy_score, confusion_matrix

# Importando classificados XGBoost
import xgboost
from xgboost import XGBClassifier

# Importando Rede Neural e skorch
import torch
import torch.nn as nn

from skorch import NeuralNetClassifier
from skorch.callbacks import EarlyStopping
from skorch.dataset import ValidSplit
from skorch.dataset import Dataset
from skorch.helper import predefined_split

In [ ]:
path_healthy = 'Healthy' #Caminho para acessar os dados saudáveis
path_damaged = 'Damaged' #Caminho para acessar os dados danificados

In [ ]:
# Função para preparar dados saudáveis
def prepare_healthy(path):
    files = ['H1.mat'] #Lista com arquivo do primeiro minuto dos dados saudáveis
    list_df_healthy = [] #Cria lista vazia
    for i in files:
        file_path = os.path.join(path, i) #Criando o caminho com a pasta e o nome do arquivo
        H = sci.io.loadmat(file_path) #Abre o arquivo .mat
        H = {k: v for k, v in H.items() if not k.startswith('__')} #Seleciona as colunas com dados
        df = pd.DataFrame({k: v.squeeze() for k, v in H.items()}) #Transforma em dataframe pandas
        df = df.drop(columns=['Speed']) #Remove coluna Speed
        list_df_healthy.append(df) #Adiciona o dataframe na lista
        print(i)
    return list_df_healthy #Retorna lista com o dataframe

# Função para preparar dados danificados
def prepare_damaged(path):
    files = ['D1.mat'] #Lista com arquivo do primeiro minuto dos dados danificados
    list_df_damaged = [] #Cria lista vazia
    for i in files:
        file_path = os.path.join(path, i) #Criando o caminho com a pasta e o nome do arquivo
        D = sci.io.loadmat(file_path) #Abre o arquivo .mat
        D = {k: v for k, v in D.items() if not k.startswith('__')} #Seleciona as colunas com dados
        df = pd.DataFrame({k: v.squeeze() for k, v in D.items()}) #Transforma em dataframe pandas
        df = df.drop(columns=['Speed', 'Torque']) #Remove as colunas Speed e Torque
        list_df_damaged.append(df) #Adiciona o dataframe na lista
        print(i)
    return list_df_damaged #Retorna lista com o dataframe

In [ ]:
healthy = prepare_healthy(path_healthy) #Chama função e cria lista com dataframes dos dados saudáveis

H1.mat


In [ ]:
damaged = prepare_damaged(path_damaged) #Chama função e cria lista com dataframes dos dados danificados

D1.mat


In [ ]:
time = np.linspace(0,60,60*40000) #Cria espaço de tempo de 0 a 60, com 2400000 valores

In [ ]:
for i in healthy:
    i['Time'] = time #Adiciona a coluna tempo

In [ ]:
for i in damaged:
    i['Time'] = time #Adiciona a coluna tempo

In [ ]:
#Função para redução de frequência com decimate do scipy

def downsample(df, freq):
    df_num = df.select_dtypes(include=[np.number]).drop(columns=['Time']) #Seleciona apenas dados numéricos, removendo a coluna tempo
    step = 40000//freq #Calcula a razão entre a frequência original e a nova

    data = {} #Cria dicionaário vazio

    #A razão é verificada se é maior do que 13
    if step<=13:
        print('Redução em uma etapa')
        for col in df_num.columns:
            data[col] = decimate(df_num[col].values, q=step, axis=0, ftype='fir', zero_phase=True) #Realiza a redução para a nova frequência
    else:
        # Para a frequência de 1 kHz, a razão é maior do que 13, sendo assim realizada em duas etapas
        print('Redução em duas etapas: 8 e 5')
        for col in df_num.columns:
            data[col] = decimate(df_num[col].values, q=8, axis=0, ftype='fir', zero_phase=True) #Primeira redução
            data[col] = decimate(data[col], q=5, axis=0, ftype='fir', zero_phase=True) #Segunda redução

    df_down=pd.DataFrame(data) #Cria dataframe com os dados reduzidos
    df_down['time'] = df['Time'].iloc[::step].values #Adiciona os dados de tempo adequados para a redução de dados

    return df_down #Retorna dataframe com os dados reduzidos

In [ ]:
#Rede Neural

class neural_network_2(nn.Module):
  def __init__(self, in_features):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
        nn.Linear(in_features,int(in_features*2)), #Cria camada de entrada
        nn.ReLU(), #Função de ativação ReLU
        nn.Linear(int(in_features*2),8), #Segunda camada
        nn.ReLU(), #Função de ativação ReLU
        nn.Linear(8,4), #Terceira camada
        nn.ReLU(), #Função de ativação ReLU
        nn.Linear(4,1), #Camada de saída
        nn.Sigmoid()) # Função sigmoíde

  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits.squeeze(-1)

In [ ]:
#Função para preparação os dados

def prepare_transform_X_y(healthy_df, damaged_df, freq, use_pca = False):
    downsample_healthy = downsample(healthy_df, freq) #Reduz os dados saudáveis de acordo com a frequência

    downsample_damaged = downsample(damaged_df, freq) #Reduz os dados danificados de acordo com a frequência

    downsample_healthy['Target'] = ['Healthy']*len(downsample_healthy) #Cria coluna de target com label saudável nos dados saudáveis

    downsample_damaged['Target'] = ['Damaged']*len(downsample_damaged) #Cria coluna de target com label danificado nos dados danificados

    df_join = pd.concat([downsample_healthy, downsample_damaged]) #Junta os dados danificados e saudáveis

    df_join = df_join.sort_values(by='time') #Ordena os valores pelo tempo

    df_test = df_join.drop(columns=['time']) #Remove a coluna de tempo

    X = df_test.iloc[:,:-1].values #Separa os dados classificadores
    y = df_test.iloc[:,-1].values #Separa os targets

    sc = StandardScaler() # Inicia normalizador
    X_norm = sc.fit_transform(X) #Normaliza os dados

    encoder = LabelEncoder() #Inicia encoder para os dados categóricos de target
    y_le = encoder.fit_transform(y) #Transforma dados categóricos em numéricos

    if use_pca == True: #Verifica se a variável use_pca é igual a True
        pca_i = PCA() # Inicia função PCA
        X_pca_i = pca_i.fit_transform(X_norm) #Transforma os dados normalizados
        comps = np.argmax(np.cumsum(pca_i.explained_variance_ratio_)>0.8)+1 #Calcula o número de componentes necessários para uma variância explicada de 80%
        pca = PCA(n_components=comps) #Inicia nova função PCA com o número de componentes
        X_pca = pca.fit_transform(X_norm) #Transforma os dados padronizados

        return [X_pca, y_le] #Retorna os dados classificadores com PCA e targets

    else:
        return [X_norm, y_le] #Retorna os dados normalizados e targets

# Função para separação em dados de treinamento e validação

def test_train_models_results(model, X_train, y_train, X_test, y_test):
    start_train = tm.perf_counter() # Calcula o tempo de início de treinamento
    model.fit(X_train, y_train) # Realiza o fit dos dados de treinamento
    end_train = tm.perf_counter() # Calcula o tempo do fim do treinamento
    time_train = end_train - start_train # Calcula o tempo de treinamento

    start_pred = tm.perf_counter() # Calcula o tempo de início de validação
    preds = model.predict(X_test) # Realiza a predição dos dados de validação
    end_pred = tm.perf_counter() # Calcula o tempo do fim da validação
    time_pred = end_pred - start_pred # Calcula o tempo de validação

    cm = confusion_matrix(y_test, preds) #Calcula a matriz de confusão
    cm = cm.tolist() #Transforma a matriz de confusão em lista
    ac = accuracy_score(y_test, preds) #Calcula a acurácia
    f1 = f1_score(y_test, preds) #Calcula a F1-Score

    results = [cm, ac, f1, time_train, time_pred]

    return results #Retorna lista com os resultados

#Função para obter os resultados

def results_freq_2(models_list, models_name_l, healthy_df, damaged_df, freq, file_export_name, pca= False):
    if pca == True:
        print('Using PCA')
    else:
        print('Not using PCA')
    X, y = prepare_transform_X_y(healthy_df, damaged_df, freq, use_pca= pca) #Obtém os dados
    print(f'Data array shape: {X.shape}')
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True) #Separa os dados em treinamento e validação, com embaralhamento dos dados,
                                                                                                            #20% de dados de validação e estado aleatório 42
    val_ds = Dataset(X_test.astype(np.float32), y_test.astype(np.float32)) #Cria dataset para a rede neural
    results = {} #Cria dicionário vazio

    for i in range(len(models_list)):
        print(f'start: {models_name_l[i]}')
        if models_name_l[i] == 'NN': #Checa se o modelo é rede neural
            models_list[i].train_split = predefined_split(val_ds) #Cria dados de treinamento e validação com o dataset definido, para utilizar parada antecipada
            models_list[i].set_params(module__in_features = np.prod(X.shape[1:])) #Define o número de features de entrada, como o número de colunas dos dados
            models_list[i].initialize() #Re-inicializa a rede neural
        results_i = test_train_models_results(models_list[i], X_train.astype(np.float32),
                                              y_train.astype(np.float32), X_test.astype(np.float32),
                                              y_test.astype(np.float32)) #Obtém os resultados para o modelo
        results[models_name_l[i]] = results_i #Adiciona os resultados no dicionário
        print(f'end: {models_name_l[i]}')
        print('--------------------------------')

    with open(file_export_name, 'w') as f:
        json.dump(results, f, indent=4) #Exporta os resultados em formato .json

    return results #Retorna os resultados

#Função para definir os modelos e parâmetros

def get_results_acc(healthy_df, damaged_df, freq, file_export_name, pca= False):
    net_2 = NeuralNetClassifier(neural_network_2,
                            criterion=nn.BCELoss(),
                            max_epochs=40, callbacks=[EarlyStopping(patience=10, threshold=0.01,load_best=True)],
                            lr=0.1, verbose=1) #Cria a rede neural com skorch, utilizando o critério de parada antecipada

    models_ml = [GaussianNB(), KNeighborsClassifier(n_neighbors = 9),
             RandomForestClassifier(criterion = 'gini', max_depth = 15, n_estimators= 150),
             SVC(C= 1, gamma= 0.1), XGBClassifier(learning_rate= 0.1, max_depth= 5, n_estimators= 200),
             net_2] #Lista com os modelos de aprendizado com os parâmetros definidos

    models_name = ['Gauss', 'KNN', 'RF', 'SVM', 'XGBoost', 'NN'] #Lista com os nomes dos modelos

    results = results_freq_2(models_ml, models_name,healthy_df,
                               damaged_df, freq = freq, pca = pca,
                               file_export_name = file_export_name) #Calacula os resultados para os modelos

    return results #Retorna os resultados

In [ ]:
results_20 = get_results_acc(healthy[0], damaged[0], freq = 20000, pca = False,
                         file_export_name = 'results_20_2.json') #Calcula os resultados para a frequência de 20 kHz, sem utilizar PCA, exportando os resultados

Not using PCA
Redução em uma etapa
Redução em uma etapa
Data array shape: (2400000, 8)
start: Gauss
end: Gauss
--------------------------------
start: KNN
end: KNN
--------------------------------
start: RF
end: RF
--------------------------------
start: SVM
end: SVM
--------------------------------
start: XGBoost
end: XGBoost
--------------------------------
start: NN
Re-initializing module because the following parameters were re-set: in_features.
Re-initializing criterion.
Re-initializing optimizer.
  epoch    train_loss    valid_acc    valid_loss      dur
-------  ------------  -----------  ------------  -------
      1        0.1645       0.9411        0.1556  32.3773
      2        0.1529       0.9424        0.1536  32.5683
      3        0.1512       0.9432        0.1520  32.4867
      4        0.1504       0.9433        0.1512  32.7390
      5        0.1499       0.9434        0.1507  32.9359
      6        0.1497       0.9435        0.1504  32.8413
      7        0.1495       

In [ ]:
results_10 = get_results_acc(healthy[0], damaged[0], freq = 10000, pca = False,
                         file_export_name = 'results_10_2.json') #Calcula os resultados para a frequência de 10 kHz, sem utilizar PCA, exportando os resultados

Not using PCA
Redução em uma etapa
Redução em uma etapa
Data array shape: (1200000, 8)
start: Gauss
end: Gauss
--------------------------------
start: KNN
end: KNN
--------------------------------
start: RF
end: RF
--------------------------------
start: SVM
end: SVM
--------------------------------
start: XGBoost
end: XGBoost
--------------------------------
start: NN
Re-initializing module because the following parameters were re-set: in_features.
Re-initializing criterion.
Re-initializing optimizer.
  epoch    train_loss    valid_acc    valid_loss      dur
-------  ------------  -----------  ------------  -------
      1        0.1821       0.9392        0.1619  16.4346
      2        0.1637       0.9400        0.1597  16.6034
      3        0.1623       0.9404        0.1588  16.5270
      4        0.1613       0.9407        0.1582  16.5813
      5        0.1607       0.9409        0.1576  16.2561
      6        0.1603       0.9408        0.1575  16.5074
      7        0.1600       

In [ ]:
results_1 = get_results_acc(healthy[0], damaged[0], freq = 1000, pca = False,
                         file_export_name = 'results_1_2.json') #Calcula os resultados para a frequência de 1 kHz, sem utilizar PCA, exportando os resultados

Not using PCA
Redução em duas etapas: 8 e 5
Redução em duas etapas: 8 e 5
Data array shape: (120000, 8)
start: Gauss
end: Gauss
--------------------------------
start: KNN
end: KNN
--------------------------------
start: RF
end: RF
--------------------------------
start: SVM
end: SVM
--------------------------------
start: XGBoost
end: XGBoost
--------------------------------
start: NN
Re-initializing module because the following parameters were re-set: in_features.
Re-initializing criterion.
Re-initializing optimizer.
  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.3933       0.9777        0.0840  1.6493
      2        0.0567       0.9855        0.0460  1.6509
      3        0.0398       0.9858        0.0399  1.6627
      4        0.0354       0.9864        0.0374  1.7169
      5        0.0332       0.9864        0.0365  1.6488
      6        0.0317       0.9871        0.0357  1.6495
      7        0.03

In [ ]:
results_20_pca = get_results_acc(healthy[0], damaged[0], freq = 20000, pca = True,
                         file_export_name = 'results_20_pca_2.json') #Calcula os resultados para a frequência de 20 kHz, utilizando PCA, exportando os resultados

Using PCA
Redução em uma etapa
Redução em uma etapa
[2026-02-14 23:01:00.266] [CUML] [warning] Warning(`fit`): As of v0.16, PCA invoked without an n_components argument defaults to using min(n_samples, n_features) rather than 1
Data array shape: (2400000, 6)
start: Gauss
end: Gauss
--------------------------------
start: KNN
end: KNN
--------------------------------
start: RF
end: RF
--------------------------------
start: SVM
end: SVM
--------------------------------
start: XGBoost
end: XGBoost
--------------------------------
start: NN
Re-initializing module because the following parameters were re-set: in_features.
Re-initializing criterion.
Re-initializing optimizer.
  epoch    train_loss    valid_acc    valid_loss      dur
-------  ------------  -----------  ------------  -------
      1        0.2427       0.9105        0.2257  32.0650
      2        0.2251       0.9115        0.2242  32.1897
      3        0.2240       0.9119        0.2233  32.5247
      4        0.2231       0.

In [ ]:
results_10_pca = get_results_acc(healthy[0], damaged[0], freq = 10000, pca = True,
                             file_export_name = 'results_10_pca_2.json') #Calcula os resultados para a frequência de 10 kHz, utilizando PCA, exportando os resultados

Using PCA
Redução em uma etapa
Redução em uma etapa
[2026-02-14 23:17:54.364] [CUML] [warning] Warning(`fit`): As of v0.16, PCA invoked without an n_components argument defaults to using min(n_samples, n_features) rather than 1
Data array shape: (1200000, 6)
start: Gauss
end: Gauss
--------------------------------
start: KNN
end: KNN
--------------------------------
start: RF
end: RF
--------------------------------
start: SVM
end: SVM
--------------------------------
start: XGBoost
end: XGBoost
--------------------------------
start: NN
Re-initializing module because the following parameters were re-set: in_features.
Re-initializing criterion.
Re-initializing optimizer.
  epoch    train_loss    valid_acc    valid_loss      dur
-------  ------------  -----------  ------------  -------
      1        0.2462       0.9104        0.2259  16.2363
      2        0.2277       0.9110        0.2242  16.3418
      3        0.2261       0.9115        0.2233  16.1117
      4        0.2252       0.

In [ ]:
results_1_pca = get_results_acc(healthy[0], damaged[0], freq = 1000, pca = True,
                            file_export_name = 'results_1_pca_2.json') #Calcula os resultados para a frequência de 1 kHz, utilizando PCA, exportando os resultados

Using PCA
Redução em duas etapas: 8 e 5
Redução em duas etapas: 8 e 5
[2026-02-14 23:23:19.773] [CUML] [warning] Warning(`fit`): As of v0.16, PCA invoked without an n_components argument defaults to using min(n_samples, n_features) rather than 1
Data array shape: (120000, 4)
start: Gauss
end: Gauss
--------------------------------
start: KNN
end: KNN
--------------------------------
start: RF
end: RF
--------------------------------
start: SVM
end: SVM
--------------------------------
start: XGBoost
end: XGBoost
--------------------------------
start: NN
Re-initializing module because the following parameters were re-set: in_features.
Re-initializing criterion.
Re-initializing optimizer.
  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.4117       0.8793        0.2773  1.6062
      2        0.2698       0.8846        0.2694  1.6077
      3        0.2665       0.8846        0.2675  1.6163
      4        0.2